# Illustration of the interactive multi-image caustic



In [1]:
!uv pip install matplotlib ipywidgets numpy lenstronomy

Using Python 3.10.18 environment at: /Users/phurailatpamhemantakumar/anaconda3/envs/ler
Audited 4 packages in 81ms


In [2]:
import io
import ipywidgets as widgets
from ipywidgets import FloatSlider, VBox, HBox, Output, Image
from IPython.display import display
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.LensModel.Solver.lens_equation_solver import LensEquationSolver
from lenstronomy.LensModel.Solver.epl_shear_solver import caustics_epl_shear
from lenstronomy.Util.param_util import phi_q2_ellipticity
import numpy as np
import matplotlib
matplotlib.use('agg')  # non-interactive backend — we render to PNG manually
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

# ---------- Custom color palette ----------
colors = {
    "black": "#000000",
    "violet": "#7E57C2",
    "brown":  "#8D6E63",
    "grey":   "#616161",
    "red":    "#E53935",
    "blue":   "#1E88E5",
    "green":  "#43A047",
    "orange": "#FB8C00",
}

# ----- Lens setup -----
lens_model_list = ["EPL", "SHEAR"]
lensModel = LensModel(lens_model_list=lens_model_list)
lens_eq_solver = LensEquationSolver(lensModel)

# ----- Create sliders -----
q_slider      = FloatSlider(value=0.6,      min=0.1,  max=0.99,   step=0.01, description='q:',      continuous_update=False)
phi_slider    = FloatSlider(value=np.pi/6,  min=0.,   max=np.pi,  step=0.05, description='φ:',    continuous_update=False)
gamma_slider  = FloatSlider(value=1.84,     min=1.5,  max=2.5,    step=0.01, description='γ:',  continuous_update=False)
gamma1_slider = FloatSlider(value=-0.05,    min=-0.2, max=0.2,    step=0.01, description='γ₁:', continuous_update=False)
gamma2_slider = FloatSlider(value=-0.05,    min=-0.2, max=0.2,    step=0.01, description='γ₂:', continuous_update=False)
beta_ra_slider  = FloatSlider(value=-0.25,  min=-1.5, max=1.5,    step=0.05, description='βₓ:',  continuous_update=False)
beta_dec_slider = FloatSlider(value=0.0,    min=-1.5, max=1.5,    step=0.05, description='βᵧ:', continuous_update=False)

# ----- Image widget to hold the plot (single render target) -----
# Fixed size: 8*150=1200px width, 6*150=900px height (matches figsize and dpi)
img_widget = Image(format='png', width='800px', height='800px')

def plot_lens_config(change=None):
    q = q_slider.value
    phi = phi_slider.value
    gamma = gamma_slider.value
    gamma1 = gamma1_slider.value
    gamma2 = gamma2_slider.value
    beta_ra = beta_ra_slider.value
    beta_dec = beta_dec_slider.value

    e1, e2 = phi_q2_ellipticity(phi, q)
    kwargs_spep = {
        'theta_E': 1.0,
        'e1': e1, 'e2': e2,
        'gamma': gamma,
        'center_x': 0.0, 'center_y': 0.0,
    }
    kwargs_shear = {'gamma1': gamma1, 'gamma2': gamma2}
    kwargs_lens = [kwargs_spep, kwargs_shear]

    fig, ax = plt.subplots(figsize=(7, 7))

    theta_ra, theta_dec = lens_eq_solver.image_position_from_source(
        sourcePos_x=beta_ra, sourcePos_y=beta_dec, kwargs_lens=kwargs_lens,
        solver="analytical", magnification_limit=1.0/100.0, arrival_time_sort=True
    )
    magnifications = np.abs(np.array(lensModel.magnification(theta_ra, theta_dec, kwargs_lens)))

    # Compute hessian properties for image type classification
    hessian = lensModel.hessian(theta_ra, theta_dec, kwargs_lens)
    determinant = np.array(
        (1 - hessian[0]) * (1 - hessian[3]) - hessian[1] * hessian[2]
    )
    trace = np.array(2 - hessian[0] - hessian[3])
    # image type classification (morse phase)
    nImages = len(theta_ra)
    image_type = np.empty(nImages, dtype=object)  # 4 images max for EPL+shear
    for j in range(nImages):
        if determinant[j] < 0:
            image_type[j] = 'II'
        elif trace[j] > 0:
            image_type[j] = 'I'
        elif trace[j] < 0:
            image_type[j] = 'III'
        else:
            image_type[j] = np.nan



    cp = caustics_epl_shear(kwargs_lens, return_which="double", maginf=-100)
    ax.plot(cp[0], cp[1], color='C1', linewidth=1.5, linestyle='--', label='Double Caustic')

    cp = caustics_epl_shear(kwargs_lens, return_which="quad", maginf=-100)
    ax.plot(cp[0], cp[1], color='C2', linewidth=1.5, linestyle='-', label='Quad Caustic')

    circle = plt.Circle((0, 0), 1.0, color='C0', fill=False,
                        linestyle='dotted', linewidth=1.5, label='Einstein Ring')
    ax.add_artist(circle)
    ax.plot(beta_ra, beta_dec, marker='x', ls='None', color='C3', label='Source', markersize=10)

    img_colors = [colors['blue'], colors['violet'], colors['brown'], colors['grey']]
    for i in range(nImages):
      ax.plot(
          theta_ra[i], theta_dec[i], marker='*', ls='None',
          color=img_colors[i % len(img_colors)],
          label=(
              f'Image {i+1}:\n'
              + f'Type {image_type[i]},\n' rf'$\mid \mu_{i+1}\mid = {magnifications[i]:.1f}$ .'
          ),
          markersize=10
      )
    # Axes & Grid
    ax.set_xlabel('x [rad]')
    ax.set_ylabel('y [rad]')
    dim_x, dim_y = 1.6, 1.6  # rectangular window
    ax.set_xlim(-dim_x, dim_x)
    ax.set_ylim(-dim_y, dim_y)
    ax.set_aspect(1, adjustable='box')
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(alpha=0.4, linestyle='--', linewidth=0.6)

    legend = ax.legend(
        handlelength=2.5, loc='center left', bbox_to_anchor=(1.03, 0.5),
        frameon=True, fontsize=12, edgecolor='lightgray', numpoints=1, scatterpoints=1
    )
    legend.get_frame().set_boxstyle('Round', pad=0.0, rounding_size=0.2)
    for handle in legend.get_lines():
        handle.set_linewidth(1.5)
        handle.set_alpha(0.85)

    ax.set_title(r'Interactive Lens Configuration ($\theta_E$ fixed at 1.0 rad)', fontsize=14)
    fig.tight_layout()

    # Render figure to PNG bytes and push into the Image widget
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    img_widget.value = buf.read()

# ----- Connect sliders -----
for s in [q_slider, phi_slider, gamma_slider, gamma1_slider, gamma2_slider, beta_ra_slider, beta_dec_slider]:
    s.observe(plot_lens_config, names='value')

# ----- Layout -----
sliders_vertical = VBox([q_slider, phi_slider, gamma_slider, gamma1_slider, gamma2_slider, beta_ra_slider, beta_dec_slider])
display(VBox([sliders_vertical, img_widget]))

# ----- Draw initial plot -----
plot_lens_config()

>**Figure :** Illustration of the interactive multi-image lens configuration showing the double-image caustic (orange, dashed) and the inner quad-image or diamond caustic (green, solid), for EPL+Shear lens model. The Einstein ring (blue, dotted) with $\theta_E=1$ is plotted for scale. The caustic geometry evolves with the lens (shape) parameters, where the axis ratio $q$ governs ellipticity and $\phi$ determines the counter-clockwise rotation relative to the x-axis. Variations in the mass density slope $\gamma$ cause the outer multi-image caustic to expand rapidly relative to the slowly decreasing inner diamond caustic, while external shear components $\gamma_1$ and $\gamma_2$ introduce skewness and rotation to the diamond structure. The resulting image positions and their absolute magnifications $|\mu|$ are displayed for the specific source location relative to the lens center. The numbering of images is based on the arrival time, with image 1 being the first to arrive at the observer. Sources positioned near the caustic boundaries in fold or cusp configurations produce high magnifications which significantly enhances the probability of detection for such lensing events.